In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Zomato Dataset.csv')

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())
print(df['Time_taken (min)'].describe())

# Plot delivery time distribution
plt.figure(figsize=(8,4))
df['Time_taken (min)'].plot(kind='hist', bins=30, color='#E24B4A', edgecolor='white')
plt.title('Distribution of Delivery Time')
plt.xlabel('Time Taken (minutes)')
plt.tight_layout()
plt.savefig('delivery_time_dist.png', dpi=150)
plt.show()

# Avg delivery time by traffic condition
plt.figure(figsize=(7,4))
df.groupby('Road_traffic_density')['Time_taken (min)'].mean().sort_values().plot(
    kind='bar', color='#378ADD', edgecolor='white')
plt.title('Avg Delivery Time by Traffic Condition')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('traffic_vs_time.png', dpi=150)
plt.show()

# Avg delivery time by weather
plt.figure(figsize=(8,4))
df.groupby('Weather_conditions')['Time_taken (min)'].mean().sort_values().plot(
    kind='bar', color='#1D9E75', edgecolor='white')
plt.title('Avg Delivery Time by Weather')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('weather_vs_time.png', dpi=150)
plt.show()

# --- Feature 1: Distance (Haversine formula) ---
# Calculates real-world km distance from GPS coordinates
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

df['distance_km'] = df.apply(lambda r: haversine(
    float(r['Restaurant_latitude']),  float(r['Restaurant_longitude']),
    float(r['Delivery_location_latitude']), float(r['Delivery_location_longitude'])
), axis=1)

# --- Feature 2: Pickup delay (minutes between order and pickup) ---
def time_to_mins(t):
    try:
        h, m = str(t).strip().split(':')
        return int(h) * 60 + int(m)
    except:
        return np.nan

df['order_mins'] = df['Time_Orderd'].apply(time_to_mins)
df['pickup_mins'] = df['Time_Order_picked'].apply(time_to_mins)
df['pickup_delay'] = df['pickup_mins'] - df['order_mins']
df['pickup_delay'] = df['pickup_delay'].apply(lambda x: x + 1440 if x < 0 else x)

# --- Feature 3: Order hour (time of day matters!) ---
df['order_hour'] = df['Time_Orderd'].apply(
    lambda t: int(str(t).strip().split(':')[0]) if ':' in str(t) else np.nan)

print("New features created:")
print(df[['distance_km','pickup_delay','order_hour']].describe())

# Drop columns not useful for prediction
drop_cols = ['ID', 'Delivery_person_ID', 'Order_Date',
             'Time_Orderd', 'Time_Order_picked',
             'Restaurant_latitude', 'Restaurant_longitude',
             'Delivery_location_latitude', 'Delivery_location_longitude']
df = df.drop(columns=drop_cols)

# Replace string 'NaN' with actual NaN
df.replace('NaN', np.nan, inplace=True)

# Fill missing values
df['Weather_conditions'].fillna(df['Weather_conditions'].mode()[0], inplace=True)
df['Road_traffic_density'].fillna(df['Road_traffic_density'].mode()[0], inplace=True)
df['Festival'].fillna('No', inplace=True)
df['City'].fillna(df['City'].mode()[0], inplace=True)
df['pickup_delay'].fillna(df['pickup_delay'].median(), inplace=True)
df['order_hour'].fillna(df['order_hour'].median(), inplace=True)

# Encode categorical columns
le = LabelEncoder()
cat_cols = ['Weather_conditions', 'Road_traffic_density', 'Type_of_order',
            'Type_of_vehicle', 'Festival', 'City']
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Convert remaining columns to numeric
df['Delivery_person_Age'] = pd.to_numeric(df['Delivery_person_Age'], errors='coerce')
df['Delivery_person_Ratings'] = pd.to_numeric(df['Delivery_person_Ratings'], errors='coerce')
df['multiple_deliveries'] = pd.to_numeric(df['multiple_deliveries'], errors='coerce')
df.fillna(df.median(numeric_only=True), inplace=True)

# Separate features and target
X = df.drop('Time_taken (min)', axis=1)
y = df['Time_taken (min)'].astype(int)

print("Ready to train! Shape:", X.shape)
print("Features:", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# --- Model 1: Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("--- Random Forest ---")
print(f"MAE:  {mean_absolute_error(y_test, rf_preds):.2f} minutes")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, rf_preds)):.2f} minutes")
print(f"R²:   {r2_score(y_test, rf_preds):.3f}")

# --- Model 2: XGBoost (main model) ---
xgb_model = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.1,
    max_depth=6, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("--- XGBoost ---")
print(f"MAE:  {mean_absolute_error(y_test, xgb_preds):.2f} minutes")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, xgb_preds)):.2f} minutes")
print(f"R²:   {r2_score(y_test, xgb_preds):.3f}")

# Feature Importance (built-in XGBoost)
feat_imp = pd.Series(xgb_model.feature_importances_, index=X.columns)
feat_imp.nlargest(10).sort_values().plot(
    kind='barh', figsize=(7,5), color='#E24B4A', edgecolor='white')
plt.title('Top 10 Features — XGBoost')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

import joblib
import os

save_path = r'C:\Users\muddu\OneDrive\Desktop\Zomato Hackathon\zomato_model.pkl'
joblib.dump(xgb_model, save_path)
print("Saved to:", save_path)
print("File exists:", os.path.exists(save_path))